# NSE Corporate Announcements Data Collection
### Do Corporate Announcements Predict Stock Market Reactions? Evidence from Indian Financial Institutions During the IL&FS Crisis

This notebook downloads every corporate announcement (metadata + PDF attachments) for 10 companies, 2017-01-01 to 2019-03-31, from NSE's corporate-announcements endpoint.

**Important NSE-specific constraints this notebook handles:**
- NSE rate-limits to **3 requests/second** -- this notebook paces requests below that.
- NSE requires session cookies obtained by visiting the homepage first, or the API returns 401/403.
- The attachment URL field in NSE's JSON is named `attchmntFile` (NSE's own naming, not a typo introduced here).
- Queries are chunked by year rather than the full 2017-2019 span in one call, to avoid any risk of a result-count limit silently truncating results (the same issue encountered with GDELT and NSE bhavcopy earlier in this project).

Fully resumable: re-running the main loop skips companies/years already completed, and skips PDFs already downloaded.

## 1. Install Libraries

In [1]:
!pip install requests pandas tqdm tenacity

## 2. Import Libraries

In [2]:
import json
import logging
import time
import os
from datetime import datetime, date
from pathlib import Path
from urllib.parse import urlencode, urljoin

import requests
import pandas as pd
from tqdm.notebook import tqdm
from tenacity import (
    retry, stop_after_attempt, wait_exponential,
    retry_if_exception_type, before_sleep_log,
)

## 3. Configuration

In [3]:
# =========================================================
# PROJECT PATHS
# =========================================================
ROOT_DIR = Path(r"C:\Users\sanka\OneDrive\Documents\AI & Trading")
ANN_DIR = ROOT_DIR / "data" / "raw" / "announcements"
LOG_DIR = ROOT_DIR / "logs"

ANN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "downloader.log"
CHECKPOINT_FILE = ANN_DIR / "checkpoint.json"


# =========================================================
# COMPANIES
# =========================================================
# folder_name: used for the local directory
# symbol: the exact NSE ticker symbol required by the API
COMPANIES = {
    "HDFC_Bank":            {"symbol": "HDFCBANK"},
    "ICICI_Bank":           {"symbol": "ICICIBANK"},
    "Axis_Bank":            {"symbol": "AXISBANK"},
    "SBI":                  {"symbol": "SBIN"},
    "Kotak_Mahindra_Bank":  {"symbol": "KOTAKBANK"},
    "Bajaj_Finance":        {"symbol": "BAJFINANCE"},
    "Bajaj_Finserv":        {"symbol": "BAJAJFINSV"},
    "Shriram_Finance":      {"symbol": "SRTRANSFIN"},
    "LIC_Housing_Finance":  {"symbol": "LICHSGFIN"},
    "PNB_Housing_Finance":  {"symbol": "PNBHOUSING"},
}


# =========================================================
# DATE RANGE
# =========================================================
START_DATE = date(2017, 1, 1)
END_DATE = date(2019, 3, 31)


# =========================================================
# NSE API SETTINGS
# =========================================================
NSE_HOME_URL = "https://www.nseindia.com"
NSE_ANNOUNCEMENTS_URL = "https://www.nseindia.com/api/corporate-announcements"
NSE_DATE_FORMAT = "%d-%m-%Y"

MIN_REQUEST_INTERVAL = 0.5
REQUEST_TIMEOUT = 20
MAX_RETRIES = 5

REQUEST_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/companies-listing/corporate-filings-announcements",
}

print("Configuration loaded.")
print("Output directory:", ANN_DIR)
print("Companies:", len(COMPANIES))
print("Date range:", START_DATE, "to", END_DATE)

Configuration loaded.
Output directory: C:\Users\sanka\OneDrive\Documents\AI & Trading\data\raw\announcements
Companies: 10
Date range: 2017-01-01 to 2019-03-31


## 4. Helper Functions

In [4]:
def safe_filename(name: str) -> str:
    """Strip characters that are invalid in Windows filenames."""
    invalid = '<>:"/\\|?*'
    for ch in invalid:
        name = name.replace(ch, "_")
    return name.strip()


def year_chunks(start: date, end: date):
    """
    Split (start, end) into per-year (chunk_start, chunk_end) date pairs.
    Querying year-by-year rather than the full multi-year span in one call
    reduces the risk of a result-count limit silently truncating results.
    """
    chunks = []
    year = start.year
    while year <= end.year:
        chunk_start = max(date(year, 1, 1), start)
        chunk_end = min(date(year, 12, 31), end)
        chunks.append((chunk_start, chunk_end))
        year += 1
    return chunks


def load_checkpoint() -> dict:
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f)
    return {}


def save_checkpoint(checkpoint: dict) -> None:
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=2)


def make_announcement_id(record: dict) -> str:
    """
    Build a stable unique key for an announcement, since NSE's JSON does not
    always include a clean single ID field across all announcement types.
    """
    symbol = record.get("symbol", "")
    bcast = record.get("an_dt", record.get("bcastRefNo", ""))
    subject = record.get("desc", record.get("attchmntText", ""))[:50]
    return f"{symbol}_{bcast}_{subject}".replace(" ", "_")

## 5. NSE Session Initialization

In [5]:
def init_session() -> requests.Session:
    """
    NSE's API rejects direct requests without valid session cookies.
    Visiting the homepage first (as a real browser would) establishes
    those cookies, which the session then carries into subsequent API calls.
    """
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)

    try:
        session.get(NSE_HOME_URL, timeout=REQUEST_TIMEOUT)
        time.sleep(1)
        session.get(
            "https://www.nseindia.com/companies-listing/corporate-filings-announcements",
            timeout=REQUEST_TIMEOUT,
        )
        time.sleep(0.5)
    except requests.RequestException as e:
        logging.warning(f"Session initialization encountered an issue: {e}")

    return session


def refresh_session(session: requests.Session) -> requests.Session:
    """Called when a 401/403 suggests cookies have expired mid-run."""
    session.cookies.clear()
    return init_session()

## 6. Build API Request

In [6]:
def build_announcements_url(symbol: str, from_date: date, to_date: date) -> str:
    """Construct the full NSE corporate-announcements API URL for one query."""
    params = {
        "index": "equities",
        "symbol": symbol,
        "from_date": from_date.strftime(NSE_DATE_FORMAT),
        "to_date": to_date.strftime(NSE_DATE_FORMAT),
    }
    return f"{NSE_ANNOUNCEMENTS_URL}?{urlencode(params)}"


NameError: name 'response' is not defined

## 7. Download JSON Metadata

In [8]:
logger = logging.getLogger("nse_downloader")

_last_request_time = [0.0]

def _pace_requests():
    """Enforce MIN_REQUEST_INTERVAL between calls to respect NSE's 3 req/s limit."""
    elapsed = time.time() - _last_request_time[0]
    if elapsed < MIN_REQUEST_INTERVAL:
        time.sleep(MIN_REQUEST_INTERVAL - elapsed)
    _last_request_time[0] = time.time()


@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=2, min=2, max=60),
    retry=retry_if_exception_type((requests.ConnectionError, requests.Timeout,
                                     json.JSONDecodeError)),
    reraise=True,
)
def fetch_announcements(session: requests.Session, symbol: str,
                          from_date: date, to_date: date) -> list:
    """
    Fetch announcement metadata for one company and date window.
    Handles 429 (backs off and retries), 401/403 (refreshes session cookies
    and retries), and malformed JSON responses.
    """
    _pace_requests()
    url = build_announcements_url(symbol, from_date, to_date)

    resp = session.get(url, timeout=REQUEST_TIMEOUT)

    if resp.status_code == 429:
        logger.warning(f"{symbol} {from_date}-{to_date}: HTTP 429, waiting 10s")
        time.sleep(10)
        raise requests.ConnectionError("Rate limited (429)")

    if resp.status_code in (401, 403):
        logger.warning(f"{symbol} {from_date}-{to_date}: HTTP {resp.status_code}, "
                       f"refreshing session")
        refresh_session(session)
        raise requests.ConnectionError(f"Auth error ({resp.status_code})")

    if resp.status_code == 404:
        logger.info(f"{symbol} {from_date}-{to_date}: HTTP 404, no data")
        return []

    if resp.status_code != 200:
        logger.error(f"{symbol} {from_date}-{to_date}: HTTP {resp.status_code} "
                     f"unexpected: {resp.text[:200]}")
        return []

    try:
        data = resp.json()
    except json.JSONDecodeError:
        logger.error(f"{symbol} {from_date}-{to_date}: JSON decode error, "
                     f"raw response: {resp.text[:200]}")
        raise

    records = data if isinstance(data, list) else data.get("data", [])
    logger.info(f"{symbol} {from_date}-{to_date}: {len(records)} announcements")
    return records

## 8. Save Metadata

In [9]:
def save_metadata(company: str, new_records: list) -> pd.DataFrame:
    """
    Append new announcement records to the company's metadata.csv,
    deduplicating by a stable announcement ID so re-running the loop
    never creates duplicate rows.
    """
    company_dir = ANN_DIR / company
    company_dir.mkdir(parents=True, exist_ok=True)
    meta_path = company_dir / "metadata.csv"

    existing_df = pd.read_csv(meta_path) if meta_path.exists() else pd.DataFrame()
    existing_ids = set(existing_df["announcement_id"]) if "announcement_id" in existing_df.columns else set()

    rows = []
    for r in new_records:
        ann_id = make_announcement_id(r)
        if ann_id in existing_ids:
            continue
        rows.append({
            "announcement_id": ann_id,
            "company": company,
            "symbol": r.get("symbol", ""),
            "issuer": r.get("sm_isin", r.get("issuer", "")),
            "broadcast_date": r.get("an_dt", ""),
            "announcement_date": r.get("date", r.get("an_dt", "")),
            "subject": r.get("desc", ""),
            "description": r.get("attchmntText", ""),
            "category": r.get("csubject", r.get("category", "")),
            "attachment_url": r.get("attchmntFile", ""),
            "local_pdf_path": "",
            "download_status": "pending",
            "download_timestamp": "",
            "_raw_json": json.dumps(r),
        })
        existing_ids.add(ann_id)

    if rows:
        new_df = pd.DataFrame(rows)
        combined = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined = existing_df

    combined.to_csv(meta_path, index=False)
    logger.info(f"{company}: {len(rows)} new metadata rows added "
               f"({len(combined)} total)")
    return combined

## 9. Download PDFs

In [10]:
@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=2, min=2, max=60),
    retry=retry_if_exception_type((requests.ConnectionError, requests.Timeout)),
    reraise=True,
)
def download_one_pdf(session: requests.Session, url: str, dest_path: Path) -> bool:
    """Download a single PDF attachment. Returns True on success."""
    _pace_requests()

    if not url.startswith("http"):
        url = urljoin(NSE_HOME_URL, url)

    try:
        resp = session.get(url, timeout=REQUEST_TIMEOUT, stream=True)
    except requests.exceptions.ChunkedEncodingError as e:
        logger.error(f"Chunked encoding error downloading {url}: {e}")
        return False

    if resp.status_code == 429:
        time.sleep(10)
        raise requests.ConnectionError("Rate limited (429) on PDF download")

    if resp.status_code in (401, 403):
        refresh_session(session)
        raise requests.ConnectionError(f"Auth error ({resp.status_code}) on PDF download")

    if resp.status_code == 404:
        logger.warning(f"404 not found: {url}")
        return False

    if resp.status_code != 200:
        logger.error(f"HTTP {resp.status_code} downloading {url}")
        return False

    dest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(dest_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    return True


def download_attachments_for_company(session: requests.Session, company: str) -> dict:
    """
    Download every attachment referenced in the company's metadata.csv.
    Skips PDFs already present on disk (resume-safe) and updates the
    metadata's download_status/local_pdf_path/download_timestamp columns.
    """
    company_dir = ANN_DIR / company
    pdf_dir = company_dir / "pdfs"
    pdf_dir.mkdir(parents=True, exist_ok=True)
    meta_path = company_dir / "metadata.csv"

    if not meta_path.exists():
        return {"downloaded": 0, "skipped": 0, "errors": 0, "no_attachment": 0}

    df = pd.read_csv(meta_path)
    counts = {"downloaded": 0, "skipped": 0, "errors": 0, "no_attachment": 0}

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{company} PDFs", leave=False):
        attachment_url = row.get("attachment_url", "")

        if pd.isna(attachment_url) or not str(attachment_url).strip():
            df.at[idx, "download_status"] = "no_attachment"
            counts["no_attachment"] += 1
            continue

        ann_id = safe_filename(str(row["announcement_id"]))[:100]
        ext = ".pdf" if str(attachment_url).lower().endswith(".pdf") else Path(str(attachment_url)).suffix or ".pdf"
        dest_path = pdf_dir / f"{ann_id}{ext}"

        if dest_path.exists() and dest_path.stat().st_size > 0:
            df.at[idx, "download_status"] = "skipped_exists"
            df.at[idx, "local_pdf_path"] = str(dest_path)
            counts["skipped"] += 1
            continue

        try:
            success = download_one_pdf(session, str(attachment_url), dest_path)
            if success:
                df.at[idx, "download_status"] = "downloaded"
                df.at[idx, "local_pdf_path"] = str(dest_path)
                df.at[idx, "download_timestamp"] = datetime.now().isoformat()
                counts["downloaded"] += 1
            else:
                df.at[idx, "download_status"] = "failed"
                counts["errors"] += 1
        except Exception as e:
            logger.error(f"{company}: failed to download {attachment_url}: {e}")
            df.at[idx, "download_status"] = "failed"
            counts["errors"] += 1

        df.to_csv(meta_path, index=False)

    return counts

## 10. Logging

In [11]:
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
print("Logging to:", LOG_FILE)

Logging to: C:\Users\sanka\OneDrive\Documents\AI & Trading\logs\downloader.log


## 11. Main Download Loop

In [12]:
def process_company(session: requests.Session, company: str, meta: dict,
                      checkpoint: dict) -> dict:
    symbol = meta["symbol"]
    chunks = year_chunks(START_DATE, END_DATE)

    company_ckpt = checkpoint.get(company, {"completed_years": [], "status": "in_progress"})

    if company_ckpt["status"] == "complete":
        print(f"{company}: metadata already complete, moving to attachments.")
    else:
        for chunk_start, chunk_end in tqdm(chunks, desc=f"{company} metadata", leave=False):
            year_label = str(chunk_start.year)
            if year_label in company_ckpt["completed_years"]:
                continue

            try:
                records = fetch_announcements(session, symbol, chunk_start, chunk_end)
                save_metadata(company, records)
            except Exception as e:
                logger.error(f"{company} {year_label}: failed after retries: {e}")
                print(f"  WARNING: {company} {year_label} failed -- see log for details")
                continue

            company_ckpt["completed_years"].append(year_label)
            checkpoint[company] = company_ckpt
            save_checkpoint(checkpoint)

        company_ckpt["status"] = "complete"
        checkpoint[company] = company_ckpt
        save_checkpoint(checkpoint)

    counts = download_attachments_for_company(session, company)

    meta_path = ANN_DIR / company / "metadata.csv"
    total_announcements = len(pd.read_csv(meta_path)) if meta_path.exists() else 0

    return {
        "company": company,
        "announcements": total_announcements,
        "pdfs_downloaded": counts["downloaded"],
        "pdfs_skipped": counts["skipped"],
        "pdfs_errors": counts["errors"],
        "no_attachment": counts["no_attachment"],
    }


checkpoint = load_checkpoint()
session = init_session()
results = []
start_time = time.time()

try:
    for company, meta in COMPANIES.items():
        print(f"\n{'='*60}\nProcessing {company} ({meta['symbol']})\n{'='*60}")
        result = process_company(session, company, meta, checkpoint)
        results.append(result)
        print(f"  Announcements: {result['announcements']} | "
              f"PDFs downloaded: {result['pdfs_downloaded']} | "
              f"skipped: {result['pdfs_skipped']} | errors: {result['pdfs_errors']}")
except KeyboardInterrupt:
    print("\nInterrupted -- progress saved to checkpoint.json and metadata CSVs. "
          "Re-run this cell to resume.")

elapsed = time.time() - start_time
print(f"\nTotal execution time: {elapsed/60:.1f} minutes")


Processing HDFC_Bank (HDFCBANK)


HDFC_Bank metadata:   0%|          | 0/3 [00:00<?, ?it/s]

HDFC_Bank PDFs:   0%|          | 0/327 [00:00<?, ?it/s]

C:\Users\sanka\AppData\Local\Temp\ipykernel_1188\1224578927.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'C:\Users\sanka\OneDrive\Documents\AI & Trading\data\raw\announcements\HDFC_Bank\pdfs\HDFCBANK_27-Dec-2017_14_08_00_Analysts_Institutional_Investor_Meet_Con._Call_Upd.zip' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "local_pdf_path"] = str(dest_path)
C:\Users\sanka\AppData\Local\Temp\ipykernel_1188\1224578927.py:85: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2026-08-01T18:01:38.549893' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "download_timestamp"] = datetime.now().isoformat()


  Announcements: 327 | PDFs downloaded: 324 | skipped: 0 | errors: 3

Processing ICICI_Bank (ICICIBANK)


...

Bajaj_Finserv PDFs:   0%|          | 0/80 [00:00<?, ?it/s]

C:\Users\sanka\AppData\Local\Temp\ipykernel_1188\1224578927.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'C:\Users\sanka\OneDrive\Documents\AI & Trading\data\raw\announcements\Bajaj_Finserv\pdfs\BAJAJFINSV_22-Dec-2017_10_44_00_Updates.zip' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "local_pdf_path"] = str(dest_path)
C:\Users\sanka\AppData\Local\Temp\ipykernel_1188\1224578927.py:85: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2026-08-01T18:22:30.060420' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "download_timestamp"] = datetime.now().isoformat()


  Announcements: 80 | PDFs downloaded: 80 | skipped: 0 | errors: 0

Processing Shriram_Finance (SRTRANSFIN)


Shriram_Finance metadata:   0%|          | 0/3 [00:00<?, ?it/s]

EmptyDataError: No columns to parse from file

In [15]:
import json

# What does the checkpoint actually say?
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r") as f:
        try:
            ckpt = json.load(f)
            print("Checkpoint is valid JSON. Contents:")
            for company, status in ckpt.items():
                print(f"  {company}: {status}")
        except json.JSONDecodeError as e:
            print(f"CHECKPOINT IS CORRUPTED: {e}")
else:
    print("No checkpoint file found at all.")

# What's actually already on disk, independent of the checkpoint?
print("\nActual data already on disk:")
for company in COMPANIES:
    meta_path = ANN_DIR / company / "metadata.csv"
    if meta_path.exists():
        df = safe_read_csv(meta_path)
        print(f"  {company}: {len(df)} rows in metadata.csv")
    else:
        print(f"  {company}: no metadata.csv yet")

Checkpoint is valid JSON. Contents:
  HDFC_Bank: {'completed_years': ['2017', '2018', '2019'], 'status': 'complete'}
  ICICI_Bank: {'completed_years': ['2017', '2018', '2019'], 'status': 'complete'}
  Axis_Bank: {'completed_years': ['2017', '2018', '2019'], 'status': 'complete'}
  SBI: {'completed_years': ['2017', '2018', '2019'], 'status': 'complete'}
  Kotak_Mahindra_Bank: {'completed_years': ['2017', '2018', '2019'], 'status': 'complete'}
  Bajaj_Finance: {'completed_years': ['2017', '2018', '2019'], 'status': 'complete'}
  Bajaj_Finserv: {'completed_years': ['2017', '2018', '2019'], 'status': 'complete'}

Actual data already on disk:


NameError: name 'safe_read_csv' is not defined

In [13]:
import shutil

# Shriram Finance's file is corrupted/empty from the crash
sf_dir = ANN_DIR / "Shriram_Finance"
if sf_dir.exists():
    shutil.rmtree(sf_dir)

# Reset its checkpoint entry so it retries cleanly under the new symbol
checkpoint = load_checkpoint()
checkpoint.pop("Shriram_Finance", None)
save_checkpoint(checkpoint)

print("Cleared Shriram Finance data and checkpoint. Ready to resume with SHRIRAMFIN.")

Cleared Shriram Finance data and checkpoint. Ready to resume with SHRIRAMFIN.


## 12. Final Summary

In [ ]:
print("=" * 70)
print("FINAL DOWNLOAD SUMMARY")
print("=" * 70)

summary_rows = []
for company in COMPANIES:
    meta_path = ANN_DIR / company / "metadata.csv"
    if meta_path.exists():
        df = pd.read_csv(meta_path)
        n_total = len(df)
        n_downloaded = (df["download_status"] == "downloaded").sum()
        n_skipped = (df["download_status"] == "skipped_exists").sum()
        n_errors = (df["download_status"] == "failed").sum()
        n_no_attach = (df["download_status"] == "no_attachment").sum()
    else:
        n_total = n_downloaded = n_skipped = n_errors = n_no_attach = 0

    summary_rows.append({
        "company": company,
        "total_announcements": n_total,
        "pdfs_downloaded": n_downloaded,
        "pdfs_skipped_existing": n_skipped,
        "pdfs_failed": n_errors,
        "no_attachment": n_no_attach,
    })
    print(f"{company:25s} | {n_total:5d} announcements | "
          f"{n_downloaded:4d} downloaded | {n_skipped:4d} skipped | "
          f"{n_errors:3d} errors | {n_no_attach:4d} no attachment")

print("=" * 70)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(ANN_DIR / "download_summary.csv", index=False)
print(f"\nSummary saved to {ANN_DIR / 'download_summary.csv'}")
summary_df